### This notebook compute "VCA15. Percentage of critical points on riverbanks or streams protected from danger" indicator for the 27 basins of IKI Project

Spanish: Porcentaje de puntos críticos en riberas de río o quebradas protegidas ante el peligro

**Created:** 10/10/2025 by Serena Gilson (sgilson@rti.org)  
**Project #:** 0219481  
**Last modified:**
**Status:** Complete and loaded in SQLite. 
**QA Status:** reviewed by  
**Original Script Stored at:** Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\Vulnerabilidad
 
**Inputs:**   Shapefile of COMIDs and districts, Puntos Criticos from ANA

**Outputs:** 
 
**Assumptions:** Given available information, calculation only includes the number of puntos criticos from ANA per area of COMID, not the number of puntos criticos 'atendidos' (from local government, etc.) as detailed in the annex. 
 
**Future work:** Normalize. 
 
**Notes:** 

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import sqlite3
import matplotlib.pyplot as plt
import re 

In [2]:
# set up user and database path
#user = 'jmayo'
#user= 'cpickering'
#user = 'sgilson'
#user = 'nreynolds'
user = 'sbakar'
#db_path = fr'C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db'
db_path = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db"
# wateralloc_db = fr"C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"
wateralloc_db = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"

In [3]:
# set up indicator ID and get scenarios from database
IndID= 518 #Indicator ID (Exposure = 2 + 0X where X is the Exposure Indicator number, Peligro= 1 +0x, VSB= 3 +0x, VSS= 4 +0x, VCA= 5 +0x)
conn = sqlite3.connect(db_path)

scenarios_df = pd.read_sql_query(
    """
    SELECT ScnID, ScnName
    FROM ScnMod
    """,
    conn
)

conn.close()

# For now: only baseline and first future
scenario_ids = scenarios_df.loc[
    scenarios_df['ScnID'].isin([1, 2]), 'ScnID'
].tolist()

# for all scenarios:
# scenario_ids = scenarios_df['ScnID'].tolist()


In [4]:
## check what scenarios are available in the WaterALLOC database
# Connect to WaterALLOC database
conn_wa = sqlite3.connect(wateralloc_db)

# Query available scenarios
scenarios_query = """
SELECT DISTINCT Scenario
FROM Scenarios
ORDER BY Scenario
"""

wa_scenarios_df = pd.read_sql_query(scenarios_query, conn_wa)

print("Available scenarios in WaterALLOC DB:")
for s in wa_scenarios_df["Scenario"]:
    print(f" - {s}")



Available scenarios in WaterALLOC DB:
 - CC_CMIP6_85_2050
 - Linea_Base_2020


In [5]:
# Extract year from scenario name and sort by year
def extract_year(name):
    match = re.search(r'\d{4}$', name)
    if match:
        return int(match.group())
    else:
        return np.nan  # just in case

wa_scenarios_df['Year'] = wa_scenarios_df['Scenario'].apply(extract_year)
wa_scenarios_df = wa_scenarios_df.sort_values('Year').reset_index(drop=True)

# Assign dynamic ScnID based on year order
wa_scenarios_df['ScnID_dynamic'] = np.arange(1, len(wa_scenarios_df) + 1)

# Map dynamic ScnID -> scenario name
scenario_mapping = dict(zip(wa_scenarios_df['ScnID_dynamic'], wa_scenarios_df['Scenario']))

print("\nDynamic scenario mapping (ScnID -> WaterALLOC Scenario Name):")
for scn_id, scn_name in scenario_mapping.items():
    print(f" - ScnID {scn_id} -> {scn_name}")


Dynamic scenario mapping (ScnID -> WaterALLOC Scenario Name):
 - ScnID 1 -> Linea_Base_2020
 - ScnID 2 -> CC_CMIP6_85_2050


In [ ]:
# define filepaths for input data
#subbasins_shapefile = fr"C:/Users/{user}/Research Triangle Institute/IKI Peru Project - General/Interno/AI2b_Modelacion/Grupos_Modelacion/GIS_WaterALLOC_General/Peru_AHD_with_districts.shp"
subbasins_shapefile = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\GIS_WaterALLOC_General\Peru_AHD_with_districts.shp"
# read in input data
subbasins_gdf = gpd.read_file(subbasins_shapefile).to_crs('EPSG:32718')

In [ ]:
# get monthly average flow from WaterALLOC database for each scenario
# (Inflow + Local Flow - Demand)
flow_all = []

conn_wa = sqlite3.connect(wateralloc_db)

for scn_id_dynamic, scenario_name in scenario_mapping.items():
    print(f"\nProcessing scenario: {scenario_name} (ScnID={scn_id_dynamic})")

    query_flow = """
    SELECT 
        a.comid AS COMID,
        AVG(
            a.[Oferta Entrada] 
          + a.[Oferta Local Sup] 
          - a.[Dem Local Sup]
        ) AS Caudal_Medio,
        COUNT(*) AS n_months
    FROM "WAMSS_Balance por COMID (+Indice de estres)" AS a
    JOIN WAMMS_RunsInfo AS b 
        ON a.RunID = b.RunID
    JOIN Scenarios AS c 
        ON c.ScnID = b.ScnID
    WHERE c.Scenario = ?
    GROUP BY a.comid
    """

    flow_df = pd.read_sql_query(query_flow, conn_wa, params=(scenario_name,))

    # Add dynamic ScnID column
    flow_df['ScnID_dynamic'] = scn_id_dynamic

    flow_all.append(flow_df)

conn_wa.close()

flow_all_df = pd.concat(flow_all, ignore_index=True)